# 07: 改善実験と知見共有

配布された**この回のフォルダ一式**を使います。ipynbの隣のPython・JSONファイルを読み込みます。
ライブラリと公開モデル・データは実行時に取得します。教材リポジトリのcloneや、隠れたコードの展開は行いません。

1. この回のフォルダをDriveに置き、このipynbをColabで開きます。
2. ランタイムを **T4 GPU** にし、最初のセルにフォルダのパスを指定します。
3. 説明→準備→設定→実行→結果確認→保存の順で進めます。

00・01はLift、02以降はLIBEROを扱います。短い接続確認と、成功率を比較する評価は区別します。
T4で短い動作確認を実施しています。長時間学習や最終評価は各実習で行います。

今回使うファイル：
- `check_environment.py`
- `common.py`
- `comparison.py`
- `config.py`
- `experiment.json`
- `experiment.py`
- `libero_env.py`
- `policy_base.py`
- `prepare.py`
- `requirements-py312.txt`
- `requirements-py313.txt`
- `robosuite_env.py`
- `rollout.py`
- `smolvla.py`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_Workshop/07_improvement"  # @param {type:"string"}
REQUIRED_FILES = [
    "check_environment.py",
    "common.py",
    "comparison.py",
    "config.py",
    "experiment.json",
    "experiment.py",
    "libero_env.py",
    "policy_base.py",
    "prepare.py",
    "requirements-py312.txt",
    "requirements-py313.txt",
    "robosuite_env.py",
    "rollout.py",
    "smolvla.py",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_07")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

この回だけの専用Pythonを作ります。初回はダウンロードに時間がかかります。教材のPythonパッケージはインストールしません。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py"), "--project", str(PROJECT)]

subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## 改善する条件を確認する

In [ ]:
print((PROJECT / "experiment.py").read_text())
# 配布例はaction chunkから実行する数を1→2に変える実験です。
# 学習したモデルで比べる場合はexperiment.pyでpolicy.pathを指定します。
# 仮説を記録し、評価用の初期状態や試行数は変更しません。

## ベースラインと候補を同じ条件で評価する

In [ ]:
result = run_python(r"""
import copy
from common import read_json
from comparison import compare
from experiment import configure
from libero_env import evaluate

baseline_config = read_json("experiment.json")
candidate_config = configure(copy.deepcopy(baseline_config))
baseline = evaluate(baseline_config, split="validation")
candidate = evaluate(candidate_config, split="validation")
compare(str(baseline), str(candidate))
""")

## 改善・悪化した初期状態を確認する

In [ ]:
comparison = json.loads((result / "comparison.json").read_text())
print(json.dumps(comparison, indent=2, ensure_ascii=False))
print("試行数、変えた条件、成功例・失敗例を一緒に報告してください。")

## 最終評価（条件を決めた後に実行する）

In [ ]:
# 候補を決めてからTrueにします。最終評価を見て条件を選び直さないでください。
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    final_result = run_python(r"""
import copy
from common import read_json
from comparison import compare
from experiment import configure
from libero_env import evaluate

baseline_config = read_json("experiment.json")
candidate_config = configure(copy.deepcopy(baseline_config))
baseline = evaluate(baseline_config, split="test")
candidate = evaluate(candidate_config, split="test")
compare(str(baseline), str(candidate))
""")
    print((final_result / "comparison.json").read_text())

## 確認課題

- 入力・出力と、変更した条件を説明できますか。
- 結果のファイルと動画を開き、失敗や不足を確認しましたか。
- 次の回で使う保存先を記録しましたか。

05では採否を確認し、06では学習・検証のエピソードを分けます。07では検証結果で候補を決めてから最終評価を行います。

## 結果をDriveに保存する

動画・ログ・モデルなど、この回のoutputsを保存します。次の回でデータ・モデルを指定できるよう、フォルダのまま保存します。保存後にランタイムを接続解除して削除してください。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_Workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "07_improvement_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)
for info in sorted(saved.glob("*/dataset/meta/info.json")):
    print("06のDATASET_ROOT:", info.parents[1])
for model in sorted(saved.glob("*/pretrained_model/config.json")):
    print("07のpolicy.path:", model.parent)